<a href="https://colab.research.google.com/github/AdiY2j/CS6910_Assignment3/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import wandb
import csv
import random
import argparse
from tqdm.notebook import tqdm
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.autograd import Variable
from matplotlib.font_manager import FontProperties
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
SOS_char = 0
EOS_char = 1
TEMP_char = 3
PAD_char = 2

In [ ]:
class Lang:
    def __init__(self, name):
        self.word2count = {'<' : 1, '>' : 1, '_' : 1, '$' : 1}
        self.word2index = {'<' : 0, '>' : 1, '_' : 2, '$' : 3}
        self.name = name
        self.index2word = {SOS_char : '<', EOS_char : '>', PAD_char : '_', TEMP_char : '$'}
        self.n_chars = 4

    def add_word(self, word):
        for c in word:
            self.add_char(c)

    def add_char(self, char):
        if char not in self.word2index: # If char not present add it in word2index and inc counter
            self.word2index[char] = self.n_chars
            self.word2count[char] = 1
            self.index2word[self.n_chars] = char
            self.n_chars += 1
        else:
            self.word2count[char] += 1 #If char already present just increment counter

In [ ]:
def prepData(dir_path, iplang, oplang):
    data = pd.read_csv(dir_path)
    data = np.array(data)

    max_ip_length = max([len(word) for word in data[:, 0]])

    max_op_length = max([len(word) for word in data[:, 1]])

    input_lang, output_lang = Lang(iplang), Lang(oplang)

    pairs = []
    for i in range(len(data)):
        pairs.append([data[i][0],data[i][1]])

    for i in range(len(pairs)):
        input_lang.add_word(pairs[i][0])
        output_lang.add_word(pairs[i][1])

    return {'pairs' : pairs,'max_input_length' : max_ip_length,'max_target_length' : max_op_length,'input_lang' : input_lang,'output_lang' : output_lang}

In [ ]:
def getWordTensor(lang, word, maxlen):
    idx = [SOS_char]
    for i in range(len(word)):
        if word[i] in lang.word2index.keys():
            idx.append(lang.word2index[word[i]])
        else:
            idx.append(TEMP_char)

    idx.append(EOS_char)
    diff = (maxlen - len(idx))
    idx.extend(([PAD_char] * diff))
    return torch.LongTensor(idx).to(device)

(26, 22, 24, 22, 28, 22)

In [ ]:
def getTensorPairs(pairs, ip_lang, op_lang, maxlen):
    tensor_pairs = []
    for data in pairs:
        tensor_pairs.append((getWordTensor(ip_lang, data[0], maxlen), getWordTensor(op_lang, data[1], maxlen)))
    return tensor_pairs

In [ ]:
def generateTensor(lang1, lang2):
    train_data = prepData('/content/drive/MyDrive/aksharantar_sampled/hin/hin_train.csv', lang1, lang2)
    val_data   = prepData('/content/drive/MyDrive/aksharantar_sampled/hin/hin_valid.csv', lang1, lang2)
    test_data  = prepData('/content/drive/MyDrive/aksharantar_sampled/hin/hin_test.csv', lang1, lang2)
    total_max_len = max([train_data['max_input_length'], train_data['max_target_length'], val_data['max_input_length'], val_data['max_target_length'], test_data['max_input_length'], test_data['max_target_length']])

    train_pairs = getTensorPairs(train_data['pairs'], train_data['input_lang'], train_data['output_lang'] , total_max_len)
    val_pairs   = getTensorPairs(val_data['pairs'], train_data['input_lang'], train_data['output_lang'], total_max_len)
    test_pairs  = getTensorPairs(test_data['pairs'], train_data['input_lang'], train_data['output_lang'], total_max_len)

    return train_pairs, val_pairs, test_pairs, train_data['input_lang'], train_data['output_lang'], total_max_len

In [ ]:
def train_batch(batch_size, num_layers, inputTensor, targetTensor, encoder, decoder, enc_optimizer, dec_optimizer, criterion, max_len, is_attention, tf_ratio = 0.5):
    loss = 0
    inputTensor = inputTensor.transpose(0, 1)
    targetTensor = targetTensor.transpose(0, 1)
    enc_hidden = encoder.initHidden(batch_size,num_layers)

    if encoder.cell_type == "LSTM":
        enc_hidden = (enc_hidden, encoder.initHidden(batch_size,num_layers))

    ip_len = inputTensor.size(0)
    op_len = targetTensor.size(0)

    enc_optimizer.zero_grad()
    dec_optimizer.zero_grad()

    num_Dir = 1
    if encoder.bidirectional :
        num_Dir = 2

    enc_outputs = torch.zeros(max_len, batch_size, encoder.hidden_size * num_Dir).to(device)

    for i in range(ip_len):
        enc_output, enc_hidden = encoder(inputTensor[i], batch_size, enc_hidden)
        enc_outputs[i] = enc_output[0]

    dec_input = torch.LongTensor([SOS_char]*batch_size).to(device)
    dec_output = None
    dec_hidden = enc_hidden

    if random.random() < tf_ratio:
        for i in range(op_len):
            if is_attention == True:
                dec_output, dec_hidden, dec_attn = decoder(dec_input, batch_size, dec_hidden, enc_outputs.reshape(batch_size, max_len, encoder.hidden_size * num_Dir))
            else:
                dec_output, dec_hidden= decoder(dec_input, batch_size, dec_hidden)
            loss += criterion(dec_output, targetTensor[i])
            dec_input = targetTensor[i]
    else:
        for i in range(op_len):
            if is_attention == True :
                dec_output, dec_hidden, dec_attn = decoder(dec_input, batch_size, dec_hidden, enc_outputs.reshape(batch_size, max_len, encoder.hidden_size * num_Dir))
            else:
                dec_output, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
            _, top_i = dec_output.data.topk(1)
            dec_input = top_i
            loss += criterion(dec_output, targetTensor[i])

    loss.backward()
    enc_optimizer.step()
    dec_optimizer.step()

    return loss.item() / op_len

array(['bindhya', 'kirankant', 'yagyopaveet', ..., 'asahmaton',
       'sulgaayin', 'anchuthengu'], dtype=object)

In [ ]:
def evaluate(batch_size, num_layers, encoder, decoder, loader, input_lang, output_lang, max_len, is_attention, test=False):
    with torch.no_grad():
        num_samples = 0
        correct_ans = 0
        actual_X = []
        actual_Y = []
        predicted_Y = []

        for inputWord, targetWord in loader:
            trans_input  = inputWord.transpose(0, 1)
            trans_output = targetWord.transpose(0, 1)
            enc_hidden = encoder.initHidden(batch_size,num_layers)
            if encoder.cell_type == "LSTM":
                enc_hidden = (enc_hidden, encoder.initHidden(batch_size,num_layers))

            ip_len = trans_input.size(0)
            op_len = trans_output.size(0)

            output = Variable(torch.LongTensor(op_len, batch_size))

            num_Dir = 1
            if encoder.bidirectional :
                num_Dir = 2

            enc_outputs = torch.zeros(max_len, batch_size, encoder.hidden_size * num_Dir).to(device)

            if test:
                for i in range(inputWord.size(0)):
                    x = [input_lang.index2word[c.item()] for c in inputWord[i] if c not in [SOS_char, EOS_char, PAD_char, TEMP_char]]
                    actual_X.append(x)

            for i in range(ip_len):
                enc_output, enc_hidden = encoder(trans_input[i], batch_size, enc_hidden)
                enc_outputs[i] = enc_output[0]

            dec_input = torch.LongTensor(([SOS_char] * batch_size)).to(device)
            dec_output = None
            dec_hidden = enc_hidden

            for i in range(op_len):
                if is_attention == True:
                    dec_output, dec_hidden, dec_attn = decoder(dec_input, batch_size, dec_hidden, enc_outputs.reshape(batch_size, max_len, encoder.hidden_size * num_Dir))
                else:
                    dec_output, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
                _, top_i = dec_output.data.topk(1)
                output[i] = torch.cat(tuple(top_i))
                dec_input = top_i

            output = output.transpose(0,1)

            output_length = output.size(0)

            for i in range(op_len):
                pred = [output_lang.index2word[c.item()] for c in output[i] if c not in [SOS_char, EOS_char, PAD_char, TEMP_char]]
                y = [output_lang.index2word[c.item()] for c in targetWord[i] if c not in [SOS_char, EOS_char, PAD_char, TEMP_char]]
                num_samples += 1
                if pred == y:
#                     print(pred, y)
                    correct_ans += 1

                if test:
                    actual_Y.append(y)
                    predicted_Y.append(sent)

    return correct_ans / num_samples

In [ ]:
def findValLoss(batch_size, num_layers, encoder, decoder, inputTensor, targetTensor, criterion, max_len, is_attention):
    with torch.no_grad():
        loss = 0
        inputTensor = inputTensor.transpose(0, 1)
        targetTensor = targetTensor.transpose(0, 1)
        enc_hidden = encoder.initHidden(batch_size,num_layers)

        if encoder.cell_type == "LSTM":
            enc_hidden = (enc_hidden, encoder.initHidden(batch_size,num_layers))

        ip_len = inputTensor.size(0)
        op_len = targetTensor.size(0)

        num_Dir = 1
        if encoder.bidirectional :
            num_Dir = 2

        enc_outputs = torch.zeros(max_len, batch_size, encoder.hidden_size * num_Dir).to(device)


        for i in range(ip_len):
            enc_output, enc_hidden = encoder(inputTensor[i], batch_size, enc_hidden)
            enc_outputs[i] = enc_output[0]

        dec_input = torch.LongTensor(([SOS_char] * batch_size)).to(device)
        dec_hidden = enc_hidden
        dec_output = None

        for i in range(op_len):
            if is_attention == True:
                dec_output, dec_hidden, dec_attn = decoder(dec_input, batch_size, dec_hidden, enc_outputs.reshape(batch_size, max_len, encoder.hidden_size * num_Dir))
            else:
                dec_output, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
            _, top_i = dec_output.data.topk(1)
            dec_input = top_i
            loss += criterion(dec_output, targetTensor[i])

    return loss.item() / op_len

(30, 68)

In [ ]:
def train(batch_size, num_layers, encoder, decoder, train_loader, val_loader, learning_rate, max_length, epochs, optimizer, input_lang, output_lang, is_attention):
    enc_optimizer = None
    criterion = nn.CrossEntropyLoss()
    dec_optimizer = None

    match optimizer:
        case "sgd":
            enc_optimizer = optim.SGD(encoder.parameters(),lr=learning_rate)
            dec_optimizer = optim.SGD(decoder.parameters(),lr=learning_rate)
        case "rmsprop":
            enc_optimizer = optim.RMSprop(encoder.parameters(),lr=learning_rate)
            dec_optimizer = optim.RMSprop(decoder.parameters(),lr=learning_rate)
        case "nadam":
            enc_optimizer = optim.NAdam(encoder.parameters(),lr=learning_rate)
            dec_optimizer = optim.NAdam(decoder.parameters(),lr=learning_rate)
        case "adam":
            enc_optimizer = optim.Adam(encoder.parameters(),lr=learning_rate)
            dec_optimizer = optim.Adam(decoder.parameters(),lr=learning_rate)

    for epoch in range(epochs):
        total_train_loss = 0
        total_val_loss = 0
        train_samples = 0
        val_samples = 0

        for ip_batch, op_batch in tqdm(train_loader):
            train_samples += 1
            loss = train_batch(batch_size, num_layers, ip_batch, op_batch, encoder, decoder, enc_optimizer, dec_optimizer, criterion, max_length, is_attention)
            total_train_loss += loss

        total_train_loss = total_train_loss / train_samples


        for ip_batch, op_batch in tqdm(val_loader):
            val_samples += 1
            loss = findValLoss(batch_size, num_layers, encoder, decoder, ip_batch, op_batch, criterion, max_length, is_attention)
            total_val_loss += loss

        total_val_loss = total_val_loss / val_samples

        train_accuracy = evaluate(batch_size, num_layers, encoder, decoder, train_loader, input_lang, output_lang, max_length, is_attention)
        val_accuracy = evaluate(batch_size, num_layers, encoder, decoder, val_loader, input_lang, output_lang, max_length, is_attention)

        print('Epoch : {}, Train Loss : {}, Train Acc : {}, Val Loss : {}, Val Acc : {}'.format(epoch+1,total_train_loss, train_accuracy, total_val_loss, val_accuracy))

        wandb.log({'Epoch' : epoch+1, 'Train Loss' : total_train_loss, 'Train Accuracy' : train_accuracy, 'Val Loss' : total_val_loss, 'Val Accuracy' : val_accuracy})

['bindhya', 'बिन्द्या']
30 68


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, embedding_size, num_layers, dropout_val, cell_type, batch_size, bidirectional=False):
        super(Encoder, self).__init__()

        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.num_layers = num_layers
        self.batch_size = batch_size
        self.cell_type = cell_type

        self.rnn = None
        self.embedding = nn.Embedding(input_size, self.embedding_size)
        self.dropout = nn.Dropout(dropout_val)
        self.bidirectional = bidirectional

        match cell_type:
            case "RNN":
                self.rnn = nn.RNN(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
            case "LSTM":
                self.rnn = nn.LSTM(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
            case "GRU":
                self.rnn = nn.GRU(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)

    def initHidden(self, batch_size, num_layers):
        num_dir = 1

        if self.bidirectional :
            num_dir = 2
        return torch.zeros(num_layers * num_dir, batch_size, self.hidden_size, device=device)

    def forward(self, input, batch_size, hidden):
        embedded = self.embedding(input).view(1,batch_size, -1)
        output, hidden = self.rnn(self.dropout(embedded), hidden)
        return output, hidden

68

In [ ]:
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size, embedding_size, num_layers, dropout_val, cell_type, batch_size, bidirectional):
        super(Decoder, self).__init__()

        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.batch_size = batch_size
        self.cell_type = cell_type
        self.dropout = nn.Dropout(dropout_val)
        self.embedding_size = embedding_size
        self.bidirectional = bidirectional
        self.rnn = None
        self.embedding = nn.Embedding(output_size, self.embedding_size)
        self.num_dir = 1

        match cell_type:
            case "RNN":
                self.rnn = nn.RNN(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
            case "LSTM":
                self.rnn = nn.LSTM(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
            case "GRU":
                self.rnn = nn.GRU(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)

        if self.bidirectional :
            self.num_dir = 2

        self.out = nn.Linear(self.hidden_size * self.num_dir, self.output_size)
        self.softmax = nn.LogSoftmax(dim = 1)


    def forward(self, input, batch_size, hidden):
        embedded = self.embedding(input).view(1,batch_size, -1)
        embedded = F.relu(self.dropout(embedded))
        output, hidden = self.rnn(embedded, hidden)
        output = self.softmax(self.out(output[0]))

        return output, hidden

[4, 5, 6, 7, 8, 7, 9, 10]
['ब', 'ि', 'न', '्', 'द', '्', 'य', 'ा']


In [ ]:
sweep_config = {
    'method' : 'bayes',
    'metric' : {
        'name' : 'Val Accuracy',
        'goal' : 'maximize'
    },
    'parameters' : {
        'epochs' : {
            'values' : [10, 15]
        },
        'hidden_size' : {
            'values' : [128, 256, 512]
        },
        'cell_type' : {
            'values' : ['GRU', 'LSTM']
        },
        'num_layers': {
            'values' : [2, 3, 4]
        },
        'batch_size' : {
            'values' : [64, 128, 256]
        },
        'drop_out' :  {
            'values' : [0, 0.1, 0.2, 0.3]
        },
        'bidirectional' : {
            'values' : [True, False]
        },
        'learning_rate' : {
            'values' : [1e-3, 5e-3, 1e-4]
        },
        'embedding_size' : {
            'values' : [128, 256, 512]
        },
        'optimizer' : {
            'values' : ['adam', 'nadam']
        },
        'attention' : {
            'values' : [True, False]
        }
    }
}

In [ ]:
sweep_id = wandb.sweep(sweep=sweep_config, project='DL_Assignment_3', entity = "cs23m009")

In [ ]:
source_lang, target_lang = 'eng', 'hin'

In [ ]:
def main():
    with wandb.init() as run:
        run_name = 'cell_{}_bs_{}_lr_{}_e_{}_nl_{}_dp_{}_bi_{}_hs_{}_es_{}_op_{}_at_{}'.format(wandb.config.cell_type, wandb.config.batch_size, wandb.config.learning_rate, wandb.config.epochs, wandb.config.num_layers, wandb.config.drop_out, wandb.config.bidirectional, wandb.config.hidden_size, wandb.config.embedding_size, wandb.config.optimizer, wandb.config.attention)
        wandb.run.name = run_name

        pairs, val_pairs, test_pairs, input_lang, output_lang, max_len  = generateTensor(source_lang, target_lang)

        encoder = Encoder(input_lang.n_chars, wandb.config['hidden_size'], wandb.config['embedding_size'], wandb.config['num_layers'], wandb.config['drop_out'], wandb.config['cell_type'], wandb.config['batch_size'], wandb.config['bidirectional']).to(device)
        decoder = Decoder(wandb.config['hidden_size'], output_lang.n_chars, wandb.config['embedding_size'], wandb.config['num_layers'], wandb.config['drop_out'], wandb.config['cell_type'], wandb.config['batch_size'], wandb.config['bidirectional']).to(device)


        train_loader = DataLoader(pairs, batch_size=wandb.config['batch_size'], shuffle=False, drop_last=True)
        val_loader = DataLoader(val_pairs, batch_size=wandb.config['batch_size'], shuffle=False, drop_last=True)
        test_loader = DataLoader(test_pairs, batch_size=wandb.config['batch_size'], shuffle=False, drop_last=True)


        temp_decoder = decoder
        if wandb.config['attention']:
            max_len += 1

        train(wandb.config['batch_size'], wandb.config['num_layers'], encoder, temp_decoder, train_loader, val_loader, wandb.config['learning_rate'], max_len, wandb.config['epochs'], wandb.config['optimizer'], input_lang, output_lang, wandb.config['attention'])

In [ ]:
wandb.agent(sweep_id, function = main, count = 10)

28

In [ ]:
wandb.finish()